<a href="https://colab.research.google.com/github/YasanthiClair/AgenticRAG/blob/main/Agentic_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from typing import List, Literal
from typing_extensions import TypedDict

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import END, StateGraph, START


class GraphState(TypedDict):
    question: str
    generation: str
    documents: List[str]



llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# --- Router Judge ---
class RouteQuery(BaseModel):
    datasource: Literal["vectorstore", "web_search"] = Field(
        description="Given a user question choose to route it to web search or a vectorstore."
    )

router_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert router directing user questions to a vectorstore or web_search. Use vectorstore for topics regarding internal knowledge base. Otherwise, use web_search."),
    ("human", "{question}")
])
question_router = router_prompt | llm.with_structured_output(RouteQuery)

# --- Document Grader Judge ---
class GradeDocuments(BaseModel):
    binary_score: Literal["yes", "no"] = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )

grader_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a grader assessing relevance of a retrieved document to a user question.\nIf the document contains keyword(s) or semantic meaning related to the question, grade it as relevant."),
    ("human", "Retrieved document: \n\n {document} \n\n User question: {question}")
])
retrieval_grader = grader_prompt | llm.with_structured_output(GradeDocuments)

# --- Hallucination Grader Judge ---
class GradeHallucinations(BaseModel):
    binary_score: Literal["yes", "no"] = Field(
        description="Answer is grounded in the facts, 'yes' or 'no'"
    )

hallucination_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts."),
    ("human", "Set of facts: \n\n {documents} \n\n LLM generation: {generation}")
])
hallucination_grader = hallucination_prompt | llm.with_structured_output(GradeHallucinations)


vectorstore = Chroma.from_texts(
    texts=["Agentic RAG uses an iterative agentic loop with self-correction mechanisms to improve retrieval quality."],
    embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()
web_search_tool = TavilySearchResults(k=3)

def retrieve(state: GraphState):
    print("--- RETRIEVING FROM VECTORSTORE ---")
    documents = retriever.invoke(state["question"])
    return {"documents": [doc.page_content for doc in documents], "question": state["question"]}

def web_search(state: GraphState):
    print("--- WEB SEARCH FALLBACK ---")
    docs = web_search_tool.invoke({"query": state["question"]})
    web_results = "\n".join([d["content"] for d in docs])
    return {"documents": [web_results], "question": state["question"]}

def generate(state: GraphState):
    print("--- GENERATING RESPONSE ---")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an assistant for question-answering tasks. Use the following context to answer the question.\nContext: {context}"),
        ("human", "{question}")
    ])
    rag_chain = prompt | llm
    response = rag_chain.invoke({"context": "\n\n".join(state["documents"]), "question": state["question"]})
    return {"generation": response.content, "documents": state["documents"], "question": state["question"]}

def grade_documents(state: GraphState):
    print("--- GRADING RETRIEVED DOCUMENTS ---")
    filtered_docs = []
    for doc in state["documents"]:
        score = retrieval_grader.invoke({"question": state["question"], "document": doc})
        if score.binary_score == "yes":
            print("   - DOCUMENT RELEVANT -")
            filtered_docs.append(doc)
        else:
            print("   - DOCUMENT NOT RELEVANT -")

    return {"documents": filtered_docs, "question": state["question"]}


def route_question(state: GraphState):
    print("--- ROUTING QUESTION ---")
    source = question_router.invoke({"question": state["question"]})
    if source.datasource == "web_search":
        print("   -> ROUTED TO WEB SEARCH")
        return "web_search"
    print("   -> ROUTED TO VECTORSTORE")
    return "vectorstore"

def decide_to_generate(state: GraphState):
    print("--- EVALUATING DOCUMENT GRADES ---")
    if not state["documents"]:
        print("   -> ALL DOCUMENTS IRRELEVANT. ROUTING TO WEB SEARCH LOOP.")
        return "web_search"
    print("   -> PROCEEDING TO GENERATION")
    return "generate"

def grade_generation_v_documents(state: GraphState):
    print("--- CHECKING FOR HALLUCINATIONS ---")
    score = hallucination_grader.invoke({"documents": "\n\n".join(state["documents"]), "generation": state["generation"]})
    if score.binary_score == "yes":
        print("   -> GENERATION IS GROUNDED. SUCCESS.")
        return "useful"
    print("   -> HALLUCINATION DETECTED! RE-RUNNING GENERATION.")
    return "not useful"

workflow = StateGraph(GraphState)

# Add Nodes
workflow.add_node("retrieve", retrieve)
workflow.add_node("web_search", web_search)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)

# Add Conditional Flows from Entry Point
workflow.add_conditional_edges(
    START,
    route_question,
    {
        "web_search": "web_search",
        "vectorstore": "retrieve",
    },
)

# Vector store results go to validation loop
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {
        "web_search": "web_search",
        "generate": "generate",
    },
)

# Web search goes straight to generation
workflow.add_edge("web_search", "generate")

# Generation goes through the Hallucination Loop validation
workflow.add_conditional_edges(
    "generate",
    grade_generation_v_documents,
    {
        "useful": END,
        "not useful": "generate", # Loop back to rewrite / regenerate if hallucinated
    },
)

# Compile Graph
app = workflow.compile()

if __name__ == "__main__":
    # Test Route 1: Vectorstore Hit
    print("\n--- TEST 1: Vectorstore Prompt ---")
    inputs = {"question": "What is Agentic RAG?"}
    for output in app.stream(inputs):
        pass

    # Test Route 2: Fallback Web Search Hit
    print("\n--- TEST 2: Web Search Fallback Prompt ---")
    inputs = {"question": "What is the weather in Paris today?"}
    for output in app.stream(inputs):
        pass